In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from numpy import exp, pi, sqrt, cos, sin


In [ ]:
def COS_solver(params_Heston, S0, K_array, t0=0, tau, COS_params, opt_type="call"):
    """
    
    Params:
    - 
    - K_array: array of strikes
    - type: str
        "call" or "put"

    Returns:
    - V: value of the option
    """
    # definition of the maturity time (we will use tau = T)
    T = tau + t0

    # simpler integration range
    [rho, kappa, gamma, bar_nu, nu0, r] = params_Heston
    [N, L] = COS_params

    a,b = -L*sqrt(T), L*sqrt(T)

    # some coefficients
    # TODO: quizás meterlos en la función de ChF_Heston
    D1 = sqrt((kappa-gamma*rho*1j*u)**2 + (u**2+1j*u)*gamma**2)
    g = (kappa-gamma*rho*1j*u-D1) / (kappa-gamma*rho*1j*u+D1)

    # define the Characteristic Function of Heston
    def ChF_Heston(u, tau, params_Heston):
        # TODO: ver qué hacer con params_Heston (sacarlos aqui o ver cómo entran)
        cc = kappa - 1j*rho*gamma*u-D1
        c1 =  exp(1j*u*tau*r + v0/gamma**2 *((1-exp(-D1*tau))/(1-g*exp(-D1*tau)))* cc)
        c2 =  exp(kappa*bar_nu/gamma**2 * (tau*cc - 2*np.log((1-g*exp(-D1*tau))/(1-g))))
        phi = c1*c2
        return phi
    
    def payoff_coeff(k):
        def chi_coeff(c,d):
            chi = 1/(1+(pi*k/(b-a))**2) * (cos(pi*k*(d-a)/(b-a))*exp(d) - cos(pi*k*(c-a)/(b-a))*exp(c) + pi*k/(b-a)*sin(pi*k*(d-a)/(b-a))*exp(d) - pi*k/(b-a)*sin(pi*k*(c-a)/(b-a))*exp(c) )
            return chi
        
        def psi_coeff(c,d):
            if k==0:
                psi = d-c
            else:
                psi =(b-a)/(pi*k) * (sin(pi*k*(d-a)/(b-a)) - sin(pi*k*(c-a)/(b-a)))
            return psi
        
        if opt_type=="call":
            H = 2/(b-a) * K_array*(chi_coeff(0,b) - psi_coeff(0,b))
        elif opt_type == "put":
            H = 2/(b-a) * K_array*(chi_coeff(a,0) - psi_coeff(a,0))

        return H
        
    # compute te sum for the final expression
    cos_sum = 0.5 * ChF_Heston(0, T, v0)*payoff_coeff[0]
    for k in range(1,N):
        U[k] = payoff_coeff()
        cos_sum += ChF_Heston(k*pi/(b-a), T, v0)*U[k]

    # compute the value of the options
    V = K* exp(-r*tau) * np.real(cos_sum * exp((X0-a)/(b-a)))

    #TODO: construir u[k] (según chat, pero no sé qué es)
    return V